In [0]:

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Importando librerias

In [0]:
import pandas as pd
import requests
import datetime

In [0]:
pd.options.display.float_format = '{:.2f}'.format  # Mostrar 2 decimales

## Funciones

In [0]:
def get_remittances_data():
    token = "5d2447bc9c11dfa56ace3d3d637670497723e627bde3319731982bfe0cc60f11"
    init_date = "1990-01-01"
    end_date = str(datetime.datetime.today().date())    
    url = "https://www.banxico.org.mx/SieAPIRest/service/v1/series/SE27803/datos/{}/{}?token={}".format(init_date, end_date, token)
    
    response = pd.DataFrame(requests.get(url).json()["bmx"]["series"][0]["datos"])
    response.columns = ["date", "remittances"]
    response["date"] = pd.to_datetime(response["date"], format = "%d/%m/%Y")  
    response["remittances"] = response["remittances"].str.replace(",", "").astype(float)
    
    return response

## Extracción de datos

In [0]:
df_remittances = get_remittances_data()

df_spark = spark.createDataFrame(df_remittances)

display(df_spark)

## Guardando datos

In [0]:
target_table = "`mx-business-intelligence-api`.bronze.remesas"

df_spark.write.saveAsTable(target_table, mode = "overwrite")

## Validación

In [0]:
%sql
SELECT * FROM `mx-business-intelligence-api`.bronze.remesas;